# SisClass-CONARQ — Sistema Especialista com Aprendizado Supervisionado
### Regressão Logística alimentando a Memória de Trabalho do sistema especialista

**Disciplina:** Sistemas Especializados e Machine Learning
**Grupo:** [nomes dos integrantes]  |  **Entrega:** 22/11/2026

**Ideia do grupo:** classificar documentos de arquivo da administração pública federal pela **Tabela de Temporalidade e Destinação de Documentos do CONARQ / Arquivo Nacional** (atividades-meio, atualização de 2024), unindo **Machine Learning** (que reconhece o assunto do documento) a um **sistema especialista** (que aplica o conhecimento da tabela e explica o resultado).

| Parte | O que faz | De onde vem o conhecimento |
|---|---|---|
| **Modelo supervisionado** (Regressão Logística) | A partir do texto do documento, prevê o fato **F1 `grupo_previsto`** (021, 022, ..., 049) | Aprende com exemplos **rotulados** de documentos públicos |
| **Sistema especialista** (motor de inferência) | Escolhe o código final, consulta prazos e destinação, aplica as regras **R1 a R10** e explica o porquê | Regras escritas a partir do **PDF do Arquivo Nacional (2024)**, a única fonte de códigos e prazos |

**Escopo:** classes 020 (Gestão de pessoas), 030 (Gestão de materiais) e 040 (Gestão de bens patrimoniais e de serviços). Fora do escopo: subclasse 080 (Pessoal militar), leitura de PDF/OCR, atividades-fim e documentos sigilosos.

⚠️ **Os dados de treino e teste são documentos públicos reais** (por exemplo, atos do Diário Oficial da União). Se a pasta `corpus/` estiver vazia, o notebook roda em **modo demonstração** com 16 textos **sintéticos** escritos à mão, e avisa. Os números do modo demonstração **não valem para o relatório**.

## 1. Estudo inicial (respostas)

**O que é aprendizado de máquina?**
É um ramo da IA em que o computador aprende padrões a partir de dados, em vez de seguir apenas instruções escritas à mão. O algoritmo ajusta um modelo com exemplos e depois usa esse modelo para fazer previsões sobre casos novos.

**O que caracteriza o aprendizado supervisionado?**
O modelo é treinado com exemplos que já têm a **resposta conhecida** (rótulo). Ele compara suas previsões com as respostas corretas e ajusta seus parâmetros para errar menos. Depois do treino, consegue prever o rótulo de documentos que nunca viu.

**O que são atributos e rótulos?**
- **Atributos (X / features):** as entradas que descrevem o caso. No nosso projeto: o **texto do documento**, transformado em números pelo **TF-IDF** (cada palavra, ou par de palavras, recebe um peso maior quando é característica do documento e menor quando aparece em todos).
- **Rótulo (y / label):** a resposta que queremos prever. No nosso projeto: o **grupo** do código de classificação (021, 022, ..., 049).

**Como classificação difere de regressão?**
- **Classificação** prevê uma **categoria**: "este documento é do grupo 022".
- **Regressão** prevê um **valor numérico**: por exemplo, quantos documentos chegam por dia.
- Atenção ao nome: a **Regressão Logística é um algoritmo de classificação**. Ela calcula uma *probabilidade* para cada grupo e escolhe o de maior probabilidade.

**Por que separar treino e teste?**
Para avaliar o modelo em documentos que ele **não viu** durante o treino. Se avaliássemos nos mesmos dados do treino, o modelo poderia "decorar" as respostas (overfitting) e a acurácia pareceria melhor do que realmente é. Aqui o **conjunto de teste (ouro)** é rotulado **à mão** pelo grupo, com o PDF aberto, e **nunca** entra no treino.

**O que é um sistema especialista?**
É um programa que reúne o conhecimento de um especialista em uma **base de regras SE…ENTÃO**, aplica essas regras com um **motor de inferência** e consegue **explicar** como chegou à conclusão (rastro "Como?") e por que pede uma informação (explicação "Por quê?"). Aqui o "especialista" é a **tabela de temporalidade do Arquivo Nacional**.

**O que é rótulo fraco?**
É um rótulo gerado automaticamente por uma regra, sem conferência humana. Usamos rótulos fracos para o **treino** (não temos tempo de rotular milhares de documentos à mão) e rótulos **à mão** para o **teste**, para que a avaliação seja honesta.

## 2. Definição do problema

| Item | Definição |
|---|---|
| **Público / usuário final** | Servidor de arquivo ou protocolo que precisa classificar documentos de gestão de pessoas, de materiais e de bens patrimoniais |
| **Entradas do modelo (atributos)** | Texto do documento, vetorizado por TF-IDF (palavras e pares de palavras) |
| **Rótulo previsto** | Fato **F1 `grupo_previsto`** (021, 022, ..., 049) |
| **Fatos vindos do conhecimento do PDF (não entram no modelo)** | F2 `candidatos` (códigos do grupo com maior semelhança entre o texto e o descritor/descrição do PDF); F3 prazos, destinação e observações da tabela de temporalidade |
| **Recomendação final** | Código sugerido + prazo corrente + prazo intermediário + destinação final (eliminação ou guarda permanente) + explicação; sem regra conclusiva ou com baixa confiança → revisão humana |

### Por que Regressão Logística?
- O grupo é uma **categoria**, exatamente o tipo de problema de **classificação** que ela resolve.
- Ela devolve uma **probabilidade** (`predict_proba`), que pode ser mostrada ao usuário: "grupo 022 com 62% de confiança".
- É **interpretável**: os coeficientes mostram quais palavras mais pesam em cada grupo, o que combina com o mecanismo de explicação do sistema especialista.
- É leve e rápida, adequada a poucos textos e a um prazo curto.

## 3. Bibliotecas

In [ ]:
# Bibliotecas são "caixas de ferramentas" prontas que usamos no código.

import re                         # re: busca padrões em textos (expressões regulares)
import unicodedata                # unicodedata: ajuda a tirar acentos das palavras
import warnings                   # warnings: esconde avisos que não atrapalham
from pathlib import Path          # pathlib: trabalha com caminhos de pastas e arquivos

import numpy as np                # numpy: faz contas e gera números aleatórios
import pandas as pd               # pandas: trabalha com tabelas (parecido com Excel)
import matplotlib.pyplot as plt   # matplotlib: desenha gráficos

# scikit-learn (sklearn): biblioteca de Aprendizado de Máquina
from sklearn.feature_extraction.text import TfidfVectorizer   # transforma texto em números (TF-IDF)
from sklearn.pipeline import make_pipeline                    # junta várias etapas em uma só
from sklearn.linear_model import LogisticRegression           # o nosso algoritmo: Regressão Logística
from sklearn.metrics import (accuracy_score,                  # calcula a % de acertos
                             f1_score,                        # média entre precisão e recall
                             confusion_matrix,                # monta a tabela de acertos e erros
                             ConfusionMatrixDisplay,          # desenha essa tabela como imagem
                             classification_report)           # gera um resumo das métricas

warnings.filterwarnings("ignore")
np.random.seed(42)                # semente: os sorteios dão sempre o mesmo resultado
pd.set_option("display.max_colwidth", 90)

print("Bibliotecas carregadas.")

## 4. Base de conhecimento extraída do PDF

**Fonte única:** Arquivo Nacional. *Código de classificação e tabela de temporalidade e destinação de documentos relativos às atividades-meio/suporte do Poder Executivo federal.* Rio de Janeiro, 2024 (Portaria AN/MGI nº 174/2024).

Dois arquivos foram extraídos do PDF por scripts (pasta `scripts/`):
- `dados/tabela_temporalidade.csv`: para cada código, o prazo no arquivo **corrente**, o prazo no **intermediário**, os eventos que disparam a contagem, a **destinação final** e as observações;
- `dados/descricoes_codigos.csv`: o descritor e a descrição ("Incluem-se documentos referentes a...") de cada código.

A extração dos prazos foi conferida automaticamente contra o texto do PDF (217 de 217 códigos finais).

In [ ]:
PASTA_DADOS = Path("dados")

tabela = pd.read_csv(PASTA_DADOS / "tabela_temporalidade.csv", dtype=str).fillna("")   # prazos e destinação
descricoes = pd.read_csv(PASTA_DADOS / "descricoes_codigos.csv", dtype=str).fillna("")  # descritor e descrição

def capitaliza(titulo):
    """Deixa 'GESTÃO DE PESSOAS' como 'Gestão de Pessoas' (palavras pequenas ficam em minúsculo)."""
    pequenas = {"de", "da", "do", "das", "dos", "e", "em", "no", "na", "nos", "nas", "a", "o", "ao", "aos", "ou", "com", "para", "por"}
    palavras = titulo.lower().split()
    return " ".join(p if (i and p in pequenas) else p.capitalize() for i, p in enumerate(palavras))

titulos = dict(zip(descricoes["codigo"], descricoes["titulo"]))       # código -> descritor
textos_desc = dict(zip(descricoes["codigo"], descricoes["texto"]))    # código -> descrição

def pais(codigo):
    """Lista o código e os seus 'pais' (031.11 -> 031.11, 031.1, 031, 030)."""
    lista, c = [], codigo
    while c:
        lista.append(c)
        if "." in c:
            c = c[:-1] if c[-2] != "." else c[:-2]        # 031.11 -> 031.1 -> 031
        elif len(c) == 3 and not c.endswith("0"):
            c = c[:2] + "0"                                 # 031 -> 030
        else:
            break
    return lista

def caminho(codigo):
    """Monta o caminho completo do código, juntando os descritores dos 'pais'.
    Ex.: 031.11 -> Gestão de Materiais > Aquisição e Incorporação > Compra > Material Permanente"""
    return " > ".join(capitaliza(titulos[c]) for c in reversed(pais(codigo)) if c in titulos)

def notas_de_remissao(texto):
    """Acha frases do PDF do tipo 'Quanto a ..., classificar no código X' (regras de exceção do especialista)."""
    return [re.sub(r"\s+", " ", m).strip() for m in re.findall(r"Quanto\s.*?código[s]?\s+\d{3}(?:\.\d+)?", texto, flags=re.S)]

# Junta tudo em uma tabela só: a BASE DE CONHECIMENTO
base = tabela.copy()
for col in ("corr_evento", "int_evento", "obs", "destinacao"):
    base[col] = base[col].str.replace(r"-\s+(?=[a-zà-ú])", "", regex=True)     # une palavras quebradas por hífen
base["grupo"]   = base["codigo"].str[:3]                                       # 3 primeiros dígitos = grupo
base["caminho"] = base["codigo"].map(caminho)
base["descricao"] = base["codigo"].map(textos_desc).fillna("")
base["notas"]   = base["descricao"].map(notas_de_remissao)
base["folha"]   = base["destinacao"].isin(["Eliminação", "Guarda permanente"])  # só códigos finais têm prazo
folhas = base[base["folha"]].reset_index(drop=True)                             # os 217 códigos finais

print(f"{len(base)} linhas na tabela; {len(folhas)} códigos finais (com prazo e destinação); {folhas['grupo'].nunique()} grupos")
print("\nCódigos finais por grupo:")
print(folhas.groupby("grupo").size().to_string())

In [ ]:
folhas[["codigo", "caminho", "corr_temp", "int_temp", "destinacao"]].head(8)

### 4.1 Do texto do PDF a números: o "casador" de descritores
Para o sistema comparar um documento com cada código, transformamos o **descritor + descrição** de cada código em vetores TF-IDF. Depois, a semelhança entre o texto do documento e cada código é medida pelo **cosseno** (0 = nada em comum; 1 = idêntico). Este é o conhecimento do especialista (o PDF) em forma numérica.

In [ ]:
def normaliza(texto):
    """Põe tudo em minúsculo, tira acentos e espaços repetidos: 'Nomeação' e 'nomeacao' viram a mesma palavra."""
    t = "".join(c for c in unicodedata.normalize("NFD", str(texto).lower()) if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", t).strip()

# Palavras muito comuns, que não ajudam a diferenciar os assuntos
PALAVRAS_COMUNS = ["a", "o", "as", "os", "de", "da", "do", "das", "dos", "e", "em", "no", "na", "nos", "nas", "um", "uma",
    "para", "por", "com", "sem", "que", "se", "ao", "aos", "à", "às", "ou", "como", "mais", "seu", "sua", "seus", "suas",
    "este", "esta", "esse", "essa", "são", "ser", "foi", "pelo", "pela", "pelos", "pelas", "art", "nº", "n", "incluem",
    "documentos", "referentes", "relativos", "subdivisões", "descritor", "classificam", "classificar", "código", "quanto",
    "órgão", "entidade", "bem", "aqueles", "não", "ano", "anos", "até", "sobre", "forma", "caso", "quando", "durante",
    "conforme", "mediante", "deste", "desta",
    # palavras de cabeçalho de atos oficiais: dizem o TIPO do ato, não o assunto (ex.: "PORTARIA ... resolve nomear")
    "portaria", "decreto", "resolucao", "resolução", "instrução", "instrucao", "normativa", "ministro", "ministra",
    "secretario", "secretário", "resolve", "considerando", "publique-se", "publique", "unica", "único", "artigo",
    "paragrafo", "parágrafo", "inciso", "dispoe", "dispõe", "fica", "ficam"]
PALAVRAS_COMUNS = sorted({normaliza(p) for p in PALAVRAS_COMUNS})

def novo_vetorizador():
    """TF-IDF com palavras e pares de palavras (ex.: 'cargo publico')."""
    return TfidfVectorizer(preprocessor=normaliza, stop_words=PALAVRAS_COMUNS, ngram_range=(1, 2), sublinear_tf=True)

# O PDF vira uma matriz: uma linha por código final
vetorizador_pdf = novo_vetorizador()
# o caminho (descritores do código e dos "pais") conta 3 vezes: é o trecho mais preciso do PDF
matriz_pdf = vetorizador_pdf.fit_transform((folhas["caminho"] + ". ") * 3 + folhas["descricao"])
vocabulario_pdf = np.array(vetorizador_pdf.get_feature_names_out())

MIN_SCORE_CODIGO = 0.06   # pontuação mínima para aceitar um código
MIN_TERMOS_CODIGO = 2     # e pelo menos 2 termos em comum com o código (uma palavra solta não decide)

def sugerir_codigos(texto, grupo=None, k=3, min_score=MIN_SCORE_CODIGO, min_termos=MIN_TERMOS_CODIGO):
    """Devolve os k códigos finais mais parecidos com o texto (opcionalmente só dentro de um grupo),
    com a pontuação e as palavras que mais contribuíram."""
    v = vetorizador_pdf.transform([texto])
    sims = (matriz_pdf @ v.T).toarray().ravel()
    if grupo is not None:
        sims = np.where(folhas["grupo"].values == grupo, sims, -1.0)     # fora do grupo: descartado
    saida = []
    for i in np.argsort(-sims)[:k]:
        if sims[i] <= 0 or sims[i] < min_score:
            continue
        comuns = v.multiply(matriz_pdf[i]).toarray().ravel()              # contribuição de cada palavra
        if (comuns > 0).sum() < min_termos:                                    # só 1 termo em comum: não decide
            continue
        palavras = [vocabulario_pdf[j] for j in np.argsort(-comuns)[:5] if comuns[j] > 0]
        saida.append({"codigo": folhas["codigo"][i], "caminho": folhas["caminho"][i], "score": float(sims[i]), "palavras": palavras})
    return saida

def pontuacao_por_grupo(texto):
    """Maior semelhança do texto com algum código de cada grupo."""
    sims = (matriz_pdf @ vetorizador_pdf.transform([texto]).T).toarray().ravel()
    out = {}
    for g, s in zip(folhas["grupo"].values, sims):
        out[g] = max(out.get(g, 0.0), float(s))
    return out

teste = "Nomear o candidato aprovado em concurso público para o cargo efetivo, com posse e exercício."
for s in sugerir_codigos(teste):
    print(f"{s['codigo']:8} score={s['score']:.2f}  {s['caminho']}")

## 5. Dataset

### Origem e critério de rotulagem
- **Documentos públicos reais** (por exemplo, atos de pessoal, materiais e patrimônio do Diário Oficial da União), um por arquivo `.txt` em `corpus/textos/`.
- **Conjunto de ouro (teste):** de 150 a 300 documentos rotulados **à mão** pelo grupo, com o PDF aberto, em `corpus/rotulos_ouro.csv` (colunas `arquivo`, `codigo`). É o equivalente ao "rótulo conferido pelo operador" do exemplo da catraca.
- **Conjunto de treino:** os demais documentos, com **rótulo fraco**: o código do PDF mais parecido com o texto, aceito só se a semelhança passar de um limiar. Para classes com poucos textos, o treino também recebe **exemplos extraídos do próprio PDF** (descritor + descrição de cada código).
- As classes **se sobrepõem de propósito** na vida real (um ato de nomeação cita "exercício" e "lotação"), então o modelo não acerta 100% e teremos erros reais para discutir.

### Atributos e rótulo
| Item | Descrição |
|---|---|
| Atributo | Texto do documento (vetorizado por TF-IDF dentro do modelo) |
| Rótulo (treino) | Grupo do código com maior semelhança (rótulo fraco) |
| Rótulo (teste) | Código conferido à mão; o grupo são os 3 primeiros dígitos |

In [ ]:
PASTA_CORPUS = Path("corpus/textos")            # textos reais (um .txt por documento)
ARQ_OURO     = Path("corpus/rotulos_ouro.csv")  # arquivo,codigo  (rotulado à mão)
PASTA_DEMO   = Path("tests/demo_sintetico")     # SÓ para testar o código; nunca usar no relatório
MIN_TEXTOS   = 30                               # abaixo disso, roda em modo demonstração

def carregar_corpus(pasta):
    """Lê todos os .txt de uma pasta e devolve uma tabela com 'arquivo' e 'texto'."""
    linhas = []
    for p in sorted(Path(pasta).glob("*.txt")):
        if p.name.upper().startswith("LEIAME"):
            continue
        t = p.read_text(encoding="utf-8", errors="ignore").strip()
        if t:
            linhas.append({"arquivo": p.name, "texto": t})
    return pd.DataFrame(linhas, columns=["arquivo", "texto"])

corpus = carregar_corpus(PASTA_CORPUS)
if ARQ_OURO.exists() and len(corpus) >= MIN_TEXTOS:
    MODO_DEMO = False
    ouro = pd.read_csv(ARQ_OURO, dtype=str)
else:
    MODO_DEMO = True
    corpus = carregar_corpus(PASTA_DEMO)
    ouro = pd.read_csv(PASTA_DEMO / "rotulos_ouro.csv", dtype=str)
    print("=" * 78)
    print("MODO DEMONSTRAÇÃO: dados SINTÉTICOS. NÃO USAR ESTES NÚMEROS NO RELATÓRIO.")
    print("=" * 78)

ouro["grupo"] = ouro["codigo"].str[:3]
print(f"Total de textos: {len(corpus)} | Conjunto de ouro (teste): {len(ouro)}")
print("\nDocumentos de ouro por grupo:")
print(ouro["grupo"].value_counts().sort_index().to_string())

In [ ]:
# Rótulo fraco: o código do PDF mais parecido com o texto (se a semelhança passar do limiar)
LIMIAR_ROTULO = 0.08

def rotular_automaticamente(df, limiar=LIMIAR_ROTULO):
    linhas = []
    for _, r in df.iterrows():
        s = sugerir_codigos(r["texto"], k=1)
        if s and s[0]["score"] >= limiar:
            linhas.append({"arquivo": r["arquivo"], "texto": r["texto"], "codigo": s[0]["codigo"],
                           "grupo": s[0]["codigo"][:3], "score": s[0]["score"]})
    return pd.DataFrame(linhas, columns=["arquivo", "texto", "codigo", "grupo", "score"])

# RN-03: nenhum documento do conjunto de ouro pode entrar no treino
corpus_treino = corpus[~corpus["arquivo"].isin(ouro["arquivo"])].reset_index(drop=True)
rotulos_fracos = rotular_automaticamente(corpus_treino)
assert set(corpus_treino["arquivo"]).isdisjoint(set(ouro["arquivo"])), "vazamento: documento de ouro no treino!"

print(f"{len(rotulos_fracos)} de {len(corpus_treino)} textos de treino receberam rótulo fraco")
if len(rotulos_fracos):
    print("\nTextos de treino por grupo (rótulo fraco):")
    print(rotulos_fracos["grupo"].value_counts().sort_index().to_string())
    if not MODO_DEMO:
        rotulos_fracos.drop(columns="texto").to_csv(PASTA_DADOS / "dataset_treino_rotulos_fracos.csv", index=False)   # vai na entrega

## 6. Separação treino / teste

- `X` contém **apenas o texto**; `y` contém o **rótulo** (grupo). O rótulo nunca entra em `X`, senão o modelo "veria a resposta".
- **Teste:** o conjunto de ouro (rotulado à mão), que o modelo nunca vê no treino.
- **Treino:** os textos restantes com rótulo fraco **mais** os exemplos extraídos do PDF.
- Diferente do exemplo da catraca (um sorteio 75/25), aqui a separação é **por origem do rótulo**: treino = rótulo automático; teste = rótulo humano. Isso evita avaliar o modelo com os mesmos critérios que o geraram.

In [ ]:
# Exemplos extraídos do próprio PDF (descritor + descrição de cada código final)
sementes = pd.DataFrame({"texto": folhas["caminho"] + ". " + folhas["descricao"], "grupo": folhas["grupo"]})

treino = pd.concat([rotulos_fracos[["texto", "grupo"]], sementes], ignore_index=True)
X_treino, y_treino = treino["texto"], treino["grupo"]

ouro_com_texto = ouro.merge(corpus, on="arquivo")          # junta o rótulo à mão com o texto
X_teste, y_teste = ouro_com_texto["texto"], ouro_com_texto["grupo"]

print("Treino:", len(X_treino), f"({len(rotulos_fracos)} textos do corpus + {len(sementes)} exemplos do PDF)")
print("Teste :", len(X_teste))

## 7. Treinamento — Regressão Logística

A regressão logística calcula uma pontuação para cada grupo a partir dos pesos das palavras e a transforma em **probabilidade**. O grupo previsto é o de maior probabilidade.

O `TfidfVectorizer` entra **dentro do pipeline**, antes do modelo: ele transforma o texto em números, e o modelo aprende a separar os grupos. `class_weight="balanced"` evita que grupos com muitos exemplos "engulam" os pequenos.

In [ ]:
# Montamos o modelo em duas etapas, uma depois da outra:
#   1) TfidfVectorizer: transforma o texto em números (peso de cada palavra)
#   2) LogisticRegression: o algoritmo que aprende a separar os grupos
modelo = make_pipeline(
    novo_vetorizador(),
    LogisticRegression(C=10, max_iter=2000, class_weight="balanced", random_state=42)
)

# fit() = TREINAR. O modelo olha os textos de treino e os grupos certos e aprende o padrão.
modelo.fit(X_treino, y_treino)

print("Modelo treinado.", len(modelo.classes_), "grupos:", list(modelo.classes_))

## 8. Avaliação — acurácia, F1 macro e matriz de confusão

In [ ]:
# predict() = PREVER. O modelo responde um grupo para cada texto de teste.
previsoes = modelo.predict(X_teste)

print(f"Acurácia no teste : {accuracy_score(y_teste, previsoes):.1%}")
print(f"F1 macro no teste : {f1_score(y_teste, previsoes, average='macro', zero_division=0):.3f}   (média do F1 de cada grupo, sem pesar pelo tamanho do grupo)\n")

# Matriz de confusão: compara o grupo CERTO (linhas) com o PREVISTO (colunas)
rotulos = sorted(set(y_teste) | set(previsoes))
matriz = confusion_matrix(y_teste, previsoes, labels=rotulos)

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(matriz, display_labels=rotulos).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=90)
ax.set_title("Teste: grupo real x grupo previsto (só o modelo de ML)")
plt.tight_layout()
plt.show()

print("Relatório por grupo:")
print(classification_report(y_teste, previsoes, digits=2, zero_division=0))

**Como ler:**
- **Acurácia** = proporção de acertos no teste. **F1 macro** = média do F1 de cada grupo; é mais exigente quando há grupos pequenos.
- Na matriz, a diagonal são os acertos. Fora dela, o modelo confundiu um grupo com outro.
- **Erro grave:** classificar um documento de **guarda permanente** em um grupo cuja destinação é **eliminação** pode levar à perda de documento que deveria ser preservado. Por isso o sistema nunca elimina nada sozinho: ele **recomenda**, mostra a fonte no PDF e pede revisão humana quando não tem confiança.

### 8.1 O ML sozinho, as regras sozinhas e os dois juntos
O fato F1 pode vir de duas fontes: do **modelo de ML** (aprende com exemplos) e das **regras do PDF** (semelhança com os descritores). O parâmetro `ALPHA` dá o peso do ML na decisão do grupo: `1.0` = só ML; `0.0` = só regras; `0.5` = os dois com o mesmo peso.

In [ ]:
ALPHA = 0.5    # peso do ML na decisão do grupo (1.0 = só ML, 0.0 = só regras)

def prever_grupo(texto, alpha=None):
    """Combina a probabilidade do ML com a semelhança bruta (cosseno) do texto com o PDF.
    Devolve (grupo, pontuação combinada). Se o texto quase não lembra nada do PDF, a pontuação das regras
    fica baixa e a confiança também."""
    alpha = ALPHA if alpha is None else alpha
    classes = list(modelo.classes_)
    p_ml = modelo.predict_proba([texto])[0]                                  # probabilidade do ML por grupo
    pont = pontuacao_por_grupo(texto)                                        # semelhança com o PDF por grupo (0 a 1)
    p_regras = np.array([pont.get(c, 0.0) for c in classes])
    p = alpha * p_ml + (1 - alpha) * p_regras                                # pontuação combinada
    i = int(np.argmax(p))
    return classes[i], float(p[i])

def avaliar_configuracao(alpha):
    """Mede grupo e código final no conjunto de ouro para um valor de ALPHA."""
    d = ouro_com_texto.copy()
    previsoes_grupo = [prever_grupo(t, alpha) for t in d["texto"]]
    d["grupo_prev"] = [g for g, _ in previsoes_grupo]
    d["confianca"] = [c for _, c in previsoes_grupo]
    codigos_prev, top3 = [], []
    for t, g in zip(d["texto"], d["grupo_prev"]):
        s = sugerir_codigos(t, grupo=g, k=3)                 # códigos do grupo previsto, do mais ao menos parecido
        codigos_prev.append(s[0]["codigo"] if s else None)
        top3.append([x["codigo"] for x in s])
    d["codigo_prev"], d["top3"] = codigos_prev, top3
    return d, {"acurácia (grupo)": accuracy_score(d["grupo"], d["grupo_prev"]),
               "F1 macro (grupo)": f1_score(d["grupo"], d["grupo_prev"], average="macro", zero_division=0),
               "acurácia (código final)": float((d["codigo"] == d["codigo_prev"]).mean()),
               "código final no top-3": float(np.mean([c in t for c, t in zip(d["codigo"], d["top3"])]))}

linhas = {}
for nome, a in [("Só ML (alpha=1,0)", 1.0), ("Só regras (alpha=0,0)", 0.0), ("ML + regras (alpha=0,5)", 0.5)]:
    linhas[nome] = avaliar_configuracao(a)[1]
pd.DataFrame(linhas).T.round(3)

In [ ]:
detalhe, metricas = avaliar_configuracao(ALPHA)

ok = metricas["acurácia (grupo)"] >= 0.80 and metricas["F1 macro (grupo)"] >= 0.75
print("Critério de aceite do projeto (acurácia >= 0,80 e F1 macro >= 0,75 no grupo):", "ATENDIDO" if ok else "NÃO ATENDIDO",
      "(modo demonstração: sem valor para o relatório)" if MODO_DEMO else "")
print("\nErros de código final (para a análise do relatório):")
detalhe[detalhe["codigo"] != detalhe["codigo_prev"]][["arquivo", "codigo", "codigo_prev", "grupo", "grupo_prev"]]

### 8.2 Calibrando o limiar de confiança
O limiar mínimo de confiança decide quando o sistema pede **revisão humana** (regra R9). A tabela mostra, para cada limiar, quantos documentos do ouro seriam enviados à revisão e qual a acurácia do grupo nos que o sistema decidiria sozinho. Escolha um limiar que reduza os erros sem mandar tudo para revisão.

In [ ]:
linhas = []
for limiar in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    decide = detalhe[detalhe["confianca"] >= limiar]                       # o sistema decidiria sozinho
    linhas.append({"limiar": limiar, "% enviados à revisão": 1 - len(decide) / len(detalhe),
                   "acurácia nos decididos": accuracy_score(decide["grupo"], decide["grupo_prev"]) if len(decide) else float("nan")})
print("Confiança média no ouro:", round(detalhe["confianca"].mean(), 3), "| mediana:", round(detalhe["confianca"].median(), 3))
pd.DataFrame(linhas).round(3)

## 9. O que o modelo aprendeu (coeficientes)

Na regressão logística, cada palavra recebe um **peso** em cada grupo:
- peso **positivo** → a palavra aumenta a chance daquele grupo;
- peso **negativo** → diminui a chance.

Esses pesos vêm **dos dados**; já as regras R1–R10 foram escritas a partir do **PDF do especialista**.

In [ ]:
# Pegamos as duas partes de dentro do pipeline: o vetorizador (vocabulário) e a regressão logística (pesos)
vetorizador = modelo.named_steps["tfidfvectorizer"]
reg_log = modelo.named_steps["logisticregression"]
vocabulario = np.array(vetorizador.get_feature_names_out())

def termos_do_grupo(grupo, n=10):
    """As n palavras (ou pares) com maior peso positivo para o grupo."""
    i = list(modelo.classes_).index(grupo)
    pesos = pd.Series(reg_log.coef_[i], index=vocabulario).sort_values(ascending=False).head(n)
    return pesos

# Mostramos o grupo mais frequente no conjunto de ouro (ou o 022, se não houver)
GRUPO_EXEMPLO = ouro["grupo"].value_counts().index[0] if len(ouro) else "022"
pesos = termos_do_grupo(GRUPO_EXEMPLO).sort_values()

print(f"Palavras que mais pesam a favor do grupo {GRUPO_EXEMPLO}:")
print(pesos.round(3).sort_values(ascending=False).to_string())

plt.figure(figsize=(7, 4))
pesos.plot(kind="barh", color="tab:green")
plt.axvline(0, color="black", lw=0.8)
plt.title(f"Palavras que mais pesam a favor do grupo {GRUPO_EXEMPLO}")
plt.tight_layout()
plt.show()

## 10. Base de Conhecimento: fatos e regras SE…ENTÃO

### Fatos do domínio (Memória de Trabalho)
| Fato | Descrição | Quem fornece |
|---|---|---|
| F0 `texto_valido` | O texto tem pelo menos 40 caracteres | Verificação da entrada |
| F1 `grupo_previsto` e `confianca` | Grupo previsto (021, 022, ..., 049) e sua probabilidade | **Modelo de Regressão Logística + pontuação do PDF** |
| F2 `candidatos` | Códigos do grupo mais parecidos com o texto (do PDF), com pontuação | Casador de descritores (PDF) |
| F3 `confianca_ok` | A confiança do grupo passa do limiar mínimo | Comparação com o limiar |
| F4 `tabela` | Prazos, eventos, destinação e observações do código | Tabela de temporalidade (PDF) |

### Regras SE…ENTÃO
| Regra | Estrutura |
|---|---|
| **R1** (Rejeitar texto inválido) | SE NOT `texto_valido` ENTÃO `documento_rejeitado` |
| **R2** (Aceitar documento) | SE `texto_valido` ENTÃO `documento_aceito` |
| **R3** (Identificar grupo) | SE `documento_aceito` E **`grupo_previsto`** ENTÃO `grupo_identificado` |
| **R4** (Escolher código) | SE `grupo_identificado` E há `candidatos` (código do grupo com pontuação mínima e ao menos 2 termos em comum com o texto) ENTÃO `codigo_escolhido` |
| **R5** (Consultar temporalidade) | SE `codigo_escolhido` ENTÃO `prazos_definidos` (corrente, intermediário, destinação) |
| **R6** (Guarda permanente) | SE `prazos_definidos` E destinação = Guarda permanente ENTÃO `alerta_guarda_permanente` |
| **R7** (Observação da tabela) | SE `prazos_definidos` E a tabela tem observação ENTÃO `exibir_observacao` |
| **R8** (Notas de remissão) | SE `codigo_escolhido` E o PDF tem nota "Quanto a..., classificar no código X" ENTÃO `exibir_notas_remissao` |
| **R9** (Revisão humana) | SE `documento_aceito` E (NOT `confianca_ok` OU não há `candidatos`) ENTÃO `revisao_humana_recomendada` |
| **R10** (Alternativas próximas) | SE `codigo_escolhido` E a diferença entre o 1º e o 2º candidato é menor que 0,02 ENTÃO `conferir_alternativas` |

**Integração:** a previsão do modelo é o fato F1, premissa da **R3**. Por consequência, ela afeta R4 a R8 (que dependem do grupo e do código escolhido).

In [ ]:
# BASE DE CONHECIMENTO: as regras do sistema especialista.
# Cada regra é um "cartão" com:
#   id    -> nome curto (R1, R2...)
#   nome  -> o que a regra faz
#   se    -> a condição escrita em texto (para mostrar na explicação)
#   cond  -> a mesma condição em código; recebe a Memória de Trabalho (m) e devolve True/False
#   entao -> a conclusão (texto) que a regra adiciona na memória quando dispara
#   acao  -> o que a regra calcula e guarda na memória (dicionário de fatos novos)

def _linha_da_tabela(codigo):
    return folhas[folhas["codigo"] == codigo].iloc[0]

def _acao_escolher_codigo(m):
    melhor = m["candidatos"][0]
    return {"codigo": melhor["codigo"], "caminho": melhor["caminho"], "palavras_regra": melhor["palavras"], "score": melhor["score"]}

def _acao_prazos(m):
    l = _linha_da_tabela(m["codigo"])
    return {"corrente": (l["corr_temp"], l["corr_evento"]), "intermediario": (l["int_temp"], l["int_evento"]),
            "destinacao": l["destinacao"], "observacao": l["obs"], "pagina_pdf": l["pagina"], "notas": list(l["notas"])}

REGRAS = [
    {"id": "R1", "nome": "Rejeitar texto inválido",
     "se": "NOT texto_valido",
     "cond": lambda m: not m["texto_valido"],
     "entao": "documento_rejeitado", "acao": lambda m: {}},

    {"id": "R2", "nome": "Aceitar documento",
     "se": "texto_valido",
     "cond": lambda m: m["texto_valido"],
     "entao": "documento_aceito", "acao": lambda m: {}},

    {"id": "R3", "nome": "Identificar grupo",
     # grupo_previsto (F1) é o fato que vem do modelo de Machine Learning
     "se": "documento_aceito E grupo_previsto",
     "cond": lambda m: m.get("documento_aceito", False) and bool(m.get("grupo_previsto")),
     "entao": "grupo_identificado", "acao": lambda m: {}},

    {"id": "R4", "nome": "Escolher código",
     "se": "grupo_identificado E existe candidato no grupo",
     "cond": lambda m: m.get("grupo_identificado", False) and len(m.get("candidatos", [])) > 0,
     "entao": "codigo_escolhido", "acao": _acao_escolher_codigo},

    {"id": "R5", "nome": "Consultar temporalidade",
     "se": "codigo_escolhido",
     "cond": lambda m: m.get("codigo_escolhido", False),
     "entao": "prazos_definidos", "acao": _acao_prazos},

    {"id": "R6", "nome": "Guarda permanente",
     "se": "prazos_definidos E destinacao = Guarda permanente",
     "cond": lambda m: m.get("prazos_definidos", False) and m["destinacao"] == "Guarda permanente",
     "entao": "alerta_guarda_permanente", "acao": lambda m: {}},

    {"id": "R7", "nome": "Observação da tabela",
     "se": "prazos_definidos E a tabela tem observação",
     "cond": lambda m: m.get("prazos_definidos", False) and m["observacao"] != "",
     "entao": "exibir_observacao", "acao": lambda m: {}},

    {"id": "R8", "nome": "Notas de remissão",
     "se": "codigo_escolhido E o PDF tem nota 'Quanto a..., classificar no código X'",
     "cond": lambda m: m.get("prazos_definidos", False) and len(m["notas"]) > 0,
     "entao": "exibir_notas_remissao", "acao": lambda m: {}},

    {"id": "R9", "nome": "Revisão humana",
     "se": "documento_aceito E (NOT confianca_ok OU sem candidatos)",
     "cond": lambda m: m.get("documento_aceito", False) and ((not m["confianca_ok"]) or len(m["candidatos"]) == 0),
     "entao": "revisao_humana_recomendada", "acao": lambda m: {}},

    {"id": "R10", "nome": "Alternativas próximas",
     "se": "codigo_escolhido E diferença entre o 1º e o 2º candidato < 0,02",
     "cond": lambda m: m.get("codigo_escolhido", False) and len(m["candidatos"]) > 1
                       and (m["candidatos"][0]["score"] - m["candidatos"][1]["score"]) < 0.02,
     "entao": "conferir_alternativas", "acao": lambda m: {}},
]

print(f"Base de conhecimento com {len(REGRAS)} regras:", [r["id"] for r in REGRAS])

## 11. Motor de inferência (encadeamento para frente) + explicação

O motor percorre a base repetidamente (**data-driven**): a cada ciclo dispara uma regra cujas condições estão satisfeitas e que ainda não foi usada, e adiciona a conclusão à Memória de Trabalho. O ciclo para quando **não há novas regras aplicáveis**.

- O rastro das regras disparadas responde à pergunta **"Como?"**.
- Se nenhuma regra chegar a um código, o sistema não tem conclusão e **pergunta ao operador**, explicando o **"Por quê?"**.

In [ ]:
def motor_inferencia(fatos):
    """MOTOR DE INFERÊNCIA (encadeamento para frente).
    Recebe os fatos iniciais, testa as regras uma a uma e vai somando conclusões
    até que nenhuma regra nova possa disparar."""

    memoria = dict(fatos)   # Memória de Trabalho: começa com os fatos iniciais
    rastro = []             # lista das regras que dispararam (para responder "Como?")
    disparadas = set()      # guarda quais regras já foram usadas, para não repetir

    mudou = True
    while mudou:                          # repete enquanto alguma regra nova disparar
        mudou = False
        for regra in REGRAS:              # olha as regras na ordem R1, R2, R3...
            # pula a regra se ela já foi usada ou se a condição não é verdadeira
            if regra["id"] in disparadas or not regra["cond"](memoria):
                continue
            # a regra DISPARA:
            disparadas.add(regra["id"])                      # marca como usada
            memoria[regra["entao"]] = True                   # adiciona a conclusão na memória
            memoria.update(regra["acao"](memoria))           # guarda também os valores calculados
            rastro.append(f"{regra['id']} ({regra['nome']}): SE {regra['se']} ENTÃO {regra['entao']}")
            mudou = True
            break                                            # volta para o início com a memória nova

    # Depois que nada mais dispara, lemos a decisão que ficou na memória
    if memoria.get("documento_rejeitado"):
        decisao = "DOCUMENTO REJEITADO: texto vazio ou muito curto"
    elif memoria.get("codigo_escolhido") and memoria.get("revisao_humana_recomendada"):
        decisao = f"CLASSIFICADO COM RESSALVA ({memoria['codigo']}): revisão humana recomendada"
    elif memoria.get("codigo_escolhido"):
        decisao = f"CLASSIFICADO: código {memoria['codigo']}"
    else:
        decisao = "NENHUMA REGRA CONCLUSIVA: perguntar ao operador"

    # Explicação "Por quê?": quando o documento é aceito mas nenhum código do grupo casou,
    # o sistema pergunta ao operador e explica o motivo da pergunta
    por_que = None
    if decisao.startswith("NENHUMA") and memoria.get("documento_aceito"):
        por_que = ('Pergunta ao operador: "Qual é o código correto para este documento?" Por quê? '
                   "Nenhum código do grupo previsto casou com o texto, e a Regra R4 precisa de um código "
                   "candidato para consultar a tabela de temporalidade.")

    # Devolve a decisão, a memória final, o rastro completo e o "Por quê?"
    return {"decisao": decisao, "memoria": memoria,
            "regras_disparadas": [r.split(" ")[0] for r in rastro],
            "rastro": rastro, "por_que": por_que}

## 12. Integração: modelo → fato F1 → regras

In [ ]:
LIMIAR_CONFIANCA = 0.10   # abaixo disso, o sistema pede revisão humana (calibrar com os dados reais: seção 8.2)
TAMANHO_MINIMO = 40       # caracteres

def formata_prazo(temp, evento):
    """Escreve o prazo do jeito que a tabela do PDF usa."""
    if temp in ("", "–"):
        return "não se aplica"
    if temp == "*":
        return f"encerra imediatamente após o evento: {evento}"
    if temp == "**":
        return f"ver observação da tabela (evento: {evento})"
    return f"{temp}, contados a partir de: {evento}"

def classificar_documento(nome, texto, limiar_confianca=LIMIAR_CONFIANCA):
    """Junta as duas partes do sistema:
    1) o MODELO olha o texto e diz qual é o grupo (fato F1);
    2) o SISTEMA ESPECIALISTA usa os fatos nas regras e decide o código, os prazos e a destinação."""

    # --- PARTE 1: Machine Learning ---
    texto_valido = len(texto.strip()) >= TAMANHO_MINIMO
    grupo, confianca, candidatos = None, 0.0, []
    if texto_valido:
        grupo, confianca = prever_grupo(texto)                       # resposta do modelo (F1)
        candidatos = sugerir_codigos(texto, grupo=grupo, k=3)         # F2: códigos do grupo mais parecidos (PDF)

    # --- PARTE 2: Sistema Especialista ---
    # Memória de Trabalho com os fatos
    fatos = {"texto_valido": texto_valido, "grupo_previsto": grupo, "confianca": confianca,
             "confianca_ok": confianca >= limiar_confianca, "candidatos": candidatos}
    resultado = motor_inferencia(fatos)                              # roda as regras R1 a R10
    m = resultado["memoria"]

    # --- Mostra tudo na tela ---
    print(f"==== {nome} ====")
    print(f"Texto (início)     : {texto[:110].strip()}{'...' if len(texto) > 110 else ''}")
    if grupo:
        nome_grupo = capitaliza(titulos.get(grupo, grupo))
        print(f"Previsão (F1)      : grupo_previsto = {grupo} ({nome_grupo})  (confiança = {confianca:.1%})")
    print(f"DECISÃO            : {resultado['decisao']}")
    if m.get("codigo_escolhido"):
        print(f"Código sugerido    : {m['codigo']} | {m['caminho']}")
        print(f"  Palavras que casaram (regra): {', '.join(m['palavras_regra']) or '-'}")
        print(f"Arquivo corrente   : {formata_prazo(*m['corrente'])}")
        print(f"Arquivo intermediário: {formata_prazo(*m['intermediario'])}")
        print(f"Destinação final   : {m['destinacao']}  (fonte: tabela de temporalidade 2024, p. {m['pagina_pdf']} do PDF)")
        if m.get("alerta_guarda_permanente"):
            print("ATENÇÃO            : guarda permanente. NÃO eliminar.")
        if m.get("exibir_observacao"):
            print(f"Observação (tabela): {m['observacao']}")
        for nota in (m["notas"] if m.get("exibir_notas_remissao") else []):
            print(f"Nota do PDF        : {nota}")
        if m.get("conferir_alternativas"):
            print("Alternativas       : " + "; ".join(f"{c['codigo']} ({c['score']:.2f})" for c in candidatos[1:]))
    print("Como? (rastro)     :")
    for passo in resultado["rastro"] or ["(nenhuma regra disparada)"]:
        print("    ", passo)
    if resultado["por_que"]:
        print(f"Por quê?           : {resultado['por_que']}")
    print("Aviso              : sistema de apoio; decisão final sujeita a revisão humana.\n")

    # Devolve o resultado para podermos comparar casos depois
    return {"grupo": grupo, "confianca": round(confianca, 3), "codigo": m.get("codigo"), **resultado}

## 13. Três casos novos (não usados no treino nem no teste)

In [ ]:
# Testamos 3 documentos NOVOS (que o modelo nunca viu).
# Os textos abaixo foram escritos pelo grupo para esta demonstração; no relatório, troque por trechos de documentos públicos reais.

# CASO 1 — Ato de provimento: nomeação de aprovado em concurso para cargo efetivo.
c1 = classificar_documento("CASO 1 — Nomeação para cargo efetivo",
    "PORTARIA. O Ministro resolve nomear o candidato aprovado em concurso público para o cargo efetivo de Técnico, "
    "com posse e entrada em exercício no prazo legal.")

# CASO 2 — Compra de material de consumo para o almoxarifado.
c2 = classificar_documento("CASO 2 — Compra de material de consumo",
    "Extrato de compra: aquisição de material de consumo para o almoxarifado (canetas, papel e material de limpeza), "
    "por pregão eletrônico, com empenho e entrega.")
# Obs.: o sistema distingue compra, doação, venda etc. pelas palavras do texto. Se o documento não traz essa pista,
# os códigos ficam empatados e a regra R10 mostra as alternativas para o operador conferir.

# CASO 3 — Texto genérico, sem assunto claro (o sistema deve pedir cautela).
c3 = classificar_documento("CASO 3 — Texto genérico",
    "Encaminho para conhecimento e providências do setor responsável, conforme solicitado anteriormente.")

### Análise dos três casos
> **Preencha após rodar com o corpus real**, usando as probabilidades e as regras que aparecem na saída acima.

**Caso 1.** Descreva o grupo previsto, a confiança, o código escolhido e o rastro de regras (R2, R3, R4, R5...). Compare os prazos e a destinação com a linha correspondente do PDF.

**Caso 2.** Idem. Observe se o sistema escolheu entre material de consumo e material permanente e se mostrou alternativas (R10).

**Caso 3.** Um texto sem assunto claro tende a ter **confiança baixa** ou **nenhum candidato**. A regra **R9** recomenda revisão humana, ou o motor responde "NENHUMA REGRA CONCLUSIVA" e explica o **Por quê?**. O sistema **não decide sozinho** quando não tem base.

## 14. Comparação: mudando apenas um fato da consulta

Mantemos o **mesmo texto** do Caso 1 (o modelo continua prevendo o mesmo grupo, com a mesma probabilidade) e mudamos **só** o fato `confianca_ok`, elevando o limiar mínimo de confiança de `LIMIAR_CONFIANCA` (0,10) para 0,95.

In [ ]:
# Repetimos o Caso 1 com o MESMO texto, mudando só o limiar de confiança exigido
texto_caso1 = ("PORTARIA. O Ministro resolve nomear o candidato aprovado em concurso público para o cargo efetivo de Técnico, "
               "com posse e entrada em exercício no prazo legal.")
c1b = classificar_documento("CASO 1b — Mesmo texto, exigindo confiança de 95%", texto_caso1, limiar_confianca=0.95)

# Tabela lado a lado para comparar os dois resultados
print("Comparação:")
print(pd.DataFrame([
    {"caso": f"1  (limiar {LIMIAR_CONFIANCA})", "grupo": c1["grupo"], "confiança": c1["confianca"],
     "regras": ", ".join(c1["regras_disparadas"]), "decisão": c1["decisao"]},
    {"caso": "1b (limiar 0,95)", "grupo": c1b["grupo"], "confiança": c1b["confianca"],
     "regras": ", ".join(c1b["regras_disparadas"]), "decisão": c1b["decisao"]},
]).to_string(index=False))
# Repare: a previsão do modelo é igual nos dois, mas a decisão muda por causa da regra R9.

A **previsão do modelo não muda** (mesmo texto → mesmo grupo e mesma probabilidade), mas a **decisão muda** de "classificado" para "classificado com ressalva: revisão humana", porque a regra **R9** passa a disparar. Isso acontece porque o limiar de confiança **não é atributo do modelo**: é um parâmetro do especialista. Mudar esse fato muda o raciocínio do sistema especialista, não o aprendizado do modelo.

## 15. Erros, limitações e representatividade

**Qual parte do programa aprende com os dados?**
A Regressão Logística (`modelo.fit`). Ela ajusta os pesos de cada palavra para cada grupo a partir dos textos de treino e dos exemplos do PDF, e com isso fornece o fato F1.

**Qual parte aplica regras humanas?**
A base `REGRAS` (R1–R10), a função `motor_inferencia` e a **tabela de temporalidade** (o conhecimento do especialista, do Arquivo Nacional). Os prazos e a destinação **não são aprendidos**: são lidos da tabela, e o sistema cita a página do PDF.

**Por que o prazo e a destinação não entram no modelo?**
Porque são **norma**: um documento de guarda permanente deve ser preservado *sempre*, não "com 90% de probabilidade". O modelo cuida apenas do que é incerto (reconhecer o assunto do texto); as regras cuidam do que é norma.

**Erros do modelo.** *(Preencha com os números reais.)* No teste, o sistema acertou ___ de ___ documentos no grupo e ___ no código final. Liste os erros mais frequentes (célula da seção 8.1) e explique: códigos com descrições parecidas (por exemplo, material permanente x material de consumo) tendem a se confundir. Registre se algum erro troca "eliminação" por "guarda permanente", que é o mais grave.

**Limitações.**
- **Rótulos de treino fracos:** gerados pelas regras do PDF, então o modelo pode herdar os vieses do casador de descritores. A avaliação usa rótulos **humanos**, justamente para evidenciar isso.
- **Conjunto de ouro pequeno:** com poucos documentos por grupo, as métricas variam bastante. Convém relatar o número de exemplos por grupo e evitar ajustar o modelo repetidamente para acertar o mesmo teste.
- **Cobertura:** só as classes 020, 030 e 040. A subclasse 080 (Pessoal militar), as atividades-fim, PDF/OCR e documentos sigilosos estão fora do escopo.
- **Atos do DOU não são o acervo de um arquivo:** o DOU publica atos (portarias, extratos), não os processos e ofícios que um órgão arquiva. O sistema mostra que a ideia funciona em textos públicos, mas não prova desempenho no acervo de um órgão real.
- **O PDF imprime "52 anos"** no arquivo intermediário dos códigos 022.1 e 022.22; o sistema reproduz o valor do documento. Vale conferir se é erro de digitação.
- O sistema **apoia** a classificação e **não substitui** o arquivista: abaixo do limiar ele pede revisão humana, e a eliminação de documentos exige o procedimento formal do órgão.

**Representatividade e ética.** Os textos devem ser **públicos** (por exemplo, atos publicados no DOU) e não podem conter dados sigilosos. Mesmo atos públicos citam nomes de pessoas, então não os redistribua além do necessário e trate os dados conforme a **LGPD**. Para uso real, seria preciso testar com documentos de vários órgãos e épocas, e medir o erro por classe.

**Um resultado alto no teste é suficiente para uso real?**
Não. Com poucos documentos e textos de um único tipo de fonte, a acurácia é instável. A decisão final deve sempre permitir **explicação** (rastro "Como?" e "Por quê?") e **revisão humana**.

## 16. Apoio à rotulagem manual do conjunto de ouro

Gera uma planilha com a sugestão do sistema (top-3) para cada texto, para o grupo **conferir e corrigir** a coluna `codigo_correto` consultando o PDF. Rotular à mão com o PDF aberto é o que garante uma avaliação honesta.

In [ ]:
def planilha_rotulagem(df, destino):
    linhas = []
    for _, r in df.iterrows():
        s = sugerir_codigos(r["texto"], k=3)
        linhas.append({"arquivo": r["arquivo"], "trecho": r["texto"][:200].replace("\n", " "),
                       "sugestao_1": s[0]["codigo"] if s else "", "sugestao_2": s[1]["codigo"] if len(s) > 1 else "",
                       "sugestao_3": s[2]["codigo"] if len(s) > 2 else "", "codigo_correto": ""})
    pd.DataFrame(linhas).to_csv(destino, index=False, encoding="utf-8-sig")
    return destino

if not MODO_DEMO:
    print("Planilha gerada em:", planilha_rotulagem(corpus, Path("corpus/planilha_rotulagem.csv")))
else:
    print("Modo demonstração: a planilha só é gerada com o corpus real.")

## Referências
- ARQUIVO NACIONAL. *Código de classificação e tabela de temporalidade e destinação de documentos relativos às atividades-meio/suporte do Poder Executivo federal.* Rio de Janeiro: Arquivo Nacional, 2024. (atualização aprovada pela Portaria AN/MGI nº 174, de 23/09/2024, publicada no DOU de 25/09/2024). **Fonte única de códigos, descritores e prazos.**
- IMPRENSA NACIONAL. *Diário Oficial da União* (fonte pública dos textos de treino e teste).
- scikit-learn. `LogisticRegression`, `TfidfVectorizer`, `make_pipeline`, métricas de classificação.

*Dados de demonstração sintéticos; os números de desempenho do relatório devem vir dos documentos públicos reais.*